# 00 · Configuración y generación de datos

Define el universo financiero, carga los hechos y verifica la cobertura temporal.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Carga de fuentes

In [2]:
actual = pd.read_csv(RAW/'fact_finance_actual.csv', parse_dates=['month'])
budget = pd.read_csv(RAW/'fact_finance_budget.csv', parse_dates=['month'])
cash = pd.read_csv(RAW/'fact_cashflow_monthly.csv', parse_dates=['month'])
routes = pd.read_csv(RAW/'dim_route.csv')
vessels = pd.read_csv(RAW/'dim_vessel.csv')
print(f'Actual: {actual.shape}; presupuesto: {budget.shape}; caja: {cash.shape}')
display(actual.head())

Actual: (252, 39); presupuesto: (252, 13); caja: (36, 15)
       month  year  ... revenue_per_available_seat_nm break_even_occupancy
0 2024-01-01  2024  ...                          0.42                 0.51
1 2024-01-01  2024  ...                          0.45                 0.57
2 2024-01-01  2024  ...                          0.29                 0.47
3 2024-01-01  2024  ...                          0.24                 0.51
4 2024-01-01  2024  ...                          0.21                 0.55

[5 rows x 39 columns]


## Perfil de cobertura

In [3]:
profile = pd.DataFrame({'dataset':['actual','budget','cash'],'rows':[len(actual),len(budget),len(cash)],'date_min':[actual.month.min(),budget.month.min(),cash.month.min()],'date_max':[actual.month.max(),budget.month.max(),cash.month.max()]})
profile.to_csv(TABLES/'00_data_profile.csv',index=False)
display(profile)

  dataset  rows   date_min   date_max
0  actual   252 2024-01-01 2026-12-01
1  budget   252 2024-01-01 2026-12-01
2    cash    36 2024-01-01 2026-12-01


## Validaciones iniciales

In [4]:
assert actual.month.nunique()==36
assert actual.route_id.nunique()==7
assert actual.revenue.gt(0).all()
assert actual.voyages.gt(0).all()
print('Cobertura completa: 36 meses × 7 rutas; importes y volúmenes positivos.')

Cobertura completa: 36 meses × 7 rutas; importes y volúmenes positivos.


## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.